# Tool-Calling LM — Experiments

Trains decoder-only Transformers from scratch (pure NumPy / PyTorch backend)
on a tool-call prediction task and compares results across six metrics.

| Model  | d_model | Heads | Layers | d_ff  | Params  |
|--------|---------|-------|--------|-------|---------|
| small  | 128     | 4     | 2      | 512   | ~700 K  |
| medium | 192     | 6     | 3      | 768   | ~2.4 M  |
| large  | 256     | 8     | 4      | 1024  | ~5.6 M  |

**All code is self-contained — no external project modules imported.**

## 0 · Choose Which Models to Run

In [ ]:
# ── Edit this list to control which models are trained and evaluated ───────────
#   Options: "small", "medium", "large"  (any subset, any order)
MODELS_TO_RUN = ["small", "medium", "large"]

PALETTE = {"small": "#4C72B0", "medium": "#DD8452", "large": "#55A868"}
print(f"Will run: {MODELS_TO_RUN}")


## 1 · Standard Imports

In [ ]:
import json, math, re, time
from pathlib import Path
import numpy as _np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

ROOT     = Path(".").resolve()
DATA_DIR = ROOT / "data_repo" / "data"


## 2 · Backend (NumPy / PyTorch MPS / CuPy CUDA)

In [ ]:
_torch = None; _cupy = None; _jax = None; _jnp = None; _jr = None

try:
    import cupy as _cupy          # type: ignore[import-untyped]
except ImportError: pass
try:
    import torch as _torch        # type: ignore[import-untyped]
except ImportError: pass
try:
    import jax as _jax            # type: ignore[import-untyped]
    import jax.numpy as _jnp      # type: ignore[import-untyped]
    import jax.random as _jr      # type: ignore[import-untyped]
except ImportError: pass

_BACKEND        = "numpy"
_active_backend = _np

class _BackendProxy:
    def __getattr__(self, name): return getattr(_active_backend, name)

xp = _BackendProxy()

def to_numpy(arr):
    if _BACKEND == "cupy"       and _cupy  is not None and isinstance(arr, _cupy.ndarray):
        return arr.get()
    if _BACKEND in ("torch_mps","torch_cuda") and _torch is not None:
        if isinstance(arr, _torch.Tensor): return arr.detach().cpu().numpy()
    return _np.asarray(arr)

def from_numpy(arr):
    arr = _np.asarray(arr)
    if _BACKEND == "cupy" and _cupy is not None: return _cupy.asarray(arr)
    if _BACKEND in ("torch_mps","torch_cuda") and _torch is not None:
        dev = "mps" if _BACKEND == "torch_mps" else "cuda"
        return _torch.from_numpy(arr).to(dev) if arr.dtype == _np.float32 \
               else _torch.as_tensor(arr).to(dev)
    return arr

def device_name():
    return {"numpy":"cpu","cupy":"cuda","torch_mps":"mps",
            "torch_cuda":"cuda","jax":"tpu/gpu/cpu"}.get(_BACKEND,"cpu")

def _s(shape): return (shape,) if isinstance(shape, int) else tuple(shape)

class _TorchRandom:
    def __init__(self, d): self._d = d
    def randn(self, *shape): return _torch.randn(*shape, device=self._d)
    def uniform(self, lo, hi, sz): return _torch.empty(sz, device=self._d).uniform_(float(lo), float(hi))
    def shuffle(self, a): _np.random.shuffle(a)

class _TorchAdd:
    def __init__(self, d): self._d = d
    def at(self, tgt, idx, vals):
        if _torch is not None and isinstance(tgt, _torch.Tensor):
            if not isinstance(idx,  _torch.Tensor): idx  = _torch.as_tensor(idx,  dtype=_torch.long,   device=self._d)
            if not isinstance(vals, _torch.Tensor): vals = _torch.as_tensor(_np.asarray(vals), dtype=tgt.dtype, device=self._d)
            fi = idx.reshape(-1); fv = vals.reshape(fi.shape[0], -1)
            tgt.scatter_add_(0, fi.unsqueeze(-1).expand_as(fv), fv)
        else: _np.add.at(tgt, idx, vals)

class _TorchBackend:
    def __init__(self, d):
        self._d = d; self.random = _TorchRandom(d); self.add = _TorchAdd(d)
        self.float32 = _torch.float32; self.int64 = _torch.int64; self.pi = float(_np.pi)

    def asarray(self, x, dtype=None): return self.array(x, dtype=dtype)
    def array(self, x, dtype=None):
        dt = self._dt(dtype)
        if _torch is not None and isinstance(x, _torch.Tensor):
            return x.to(dtype=dt, device=self._d) if dt else x.to(device=self._d)
        na = _np.asarray(x)
        return _torch.as_tensor(na, dtype=dt, device=self._d) if dt else _torch.as_tensor(na, device=self._d)

    def zeros(self, sh, dtype=None): return _torch.zeros(_s(sh), dtype=self._dt(dtype) or _torch.float32, device=self._d)
    def ones(self,  sh, dtype=None): return _torch.ones( _s(sh), dtype=self._dt(dtype) or _torch.float32, device=self._d)
    def full(self, sh, v, dtype=None): return _torch.full(_s(sh), v, dtype=self._dt(dtype) or _torch.float32, device=self._d)
    def zeros_like(self, a): return _torch.zeros_like(a) if isinstance(a, _torch.Tensor) else _torch.zeros(a.shape, dtype=_torch.float32, device=self._d)
    def ones_like(self,  a): return _torch.ones_like(a)  if isinstance(a, _torch.Tensor) else _torch.ones( a.shape, dtype=_torch.float32, device=self._d)
    def arange(self, *args): return _torch.arange(*args, device=self._d)

    def exp(self,  x): return _torch.exp( self._t(x))
    def log(self,  x): return _torch.log( self._t(x))
    def sin(self,  x): return _torch.sin( self._t(x))
    def cos(self,  x): return _torch.cos( self._t(x))
    def tanh(self, x): return _torch.tanh(self._t(x))
    def sqrt(self, x): return float(_np.sqrt(x)) if isinstance(x,(int,float)) else _torch.sqrt(self._t(x))

    def sum( self, x, axis=None, keepdims=False): x=self._t(x); return x.sum()  if axis is None else x.sum( dim=axis, keepdim=keepdims)
    def max( self, x, axis=None, keepdims=False): x=self._t(x); return x.max()  if axis is None else x.max( dim=axis, keepdim=keepdims).values
    def min( self, x, axis=None, keepdims=False): x=self._t(x); return x.min()  if axis is None else x.min( dim=axis, keepdim=keepdims).values
    def mean(self, x, axis=None, keepdims=False): x=self._t(x,dtype=_torch.float32); return x.mean() if axis is None else x.mean(dim=axis, keepdim=keepdims)

    def swapaxes(self, x, a, b):   return self._t(x).transpose(a, b)
    def transpose(self, x, axes):  return self._t(x).permute(*axes)
    def broadcast_to(self, x, sh): return self._t(x).expand(sh).clone()
    def expand_dims(self, x, ax):  return self._t(x).unsqueeze(ax)
    def triu(self, x, k=0):        return _torch.triu(self._t(x), diagonal=k)

    def _t(self, x, dtype=None):
        if _torch is not None and isinstance(x, _torch.Tensor):
            return x.to(device=self._d, dtype=dtype) if dtype else x.to(device=self._d)
        return _torch.as_tensor(_np.asarray(x), dtype=dtype, device=self._d)
    def _dt(self, dt):
        if dt is None: return None
        if _torch is not None and dt is _torch.float32: return _torch.float32
        if _torch is not None and dt is _torch.int64:   return _torch.int64
        if dt == _np.float32: return _torch.float32
        if dt == _np.int64:   return _torch.int64
        return None

def _try_cupy():
    global _active_backend, _BACKEND
    if _cupy is None: return False
    try: _cupy.array([1.0]); _active_backend, _BACKEND = _cupy, "cupy"; return True
    except: return False

def _try_torch(dev):
    global _active_backend, _BACKEND
    if _torch is None: return False
    try:
        if dev == "mps":
            if not (hasattr(_torch.backends,"mps") and _torch.backends.mps.is_available()): return False
            _active_backend, _BACKEND = _TorchBackend("mps"), "torch_mps"
        else:
            if not _torch.cuda.is_available(): return False
            _active_backend, _BACKEND = _TorchBackend("cuda"), "torch_cuda"
        return True
    except: return False

def set_backend(name="auto"):
    global _active_backend, _BACKEND
    if name == "cpu":   _active_backend, _BACKEND = _np, "numpy"; return
    if name in ("cuda","gpu"):
        if _try_cupy() or _try_torch("cuda"): return
        raise RuntimeError("No CUDA backend found.")
    if name == "mps":
        if _try_torch("mps"): return
        raise RuntimeError("MPS not available.")
    if name == "auto":
        if _try_cupy(): return
        if _try_torch("mps"): return
        if _try_torch("cuda"): return
        _active_backend, _BACKEND = _np, "numpy"

def print_backend_info():
    bn = type(_active_backend).__name__ if _BACKEND != "numpy" else "numpy"
    print(f"Backend: {_BACKEND}  |  device: {device_name()}  |  xp: {bn}")


In [ ]:
DEVICE = "auto"   # change to 'cpu' / 'mps' / 'cuda' if needed
set_backend(DEVICE)
print_backend_info()


## 3 · Autograd Engine

In [ ]:
def _sum_to_shape(x, shape):
    while x.ndim > len(shape): x = x.sum(axis=0)
    for i, s in enumerate(shape):
        if s == 1 and x.shape[i] != 1: x = x.sum(axis=i, keepdims=True)
    return x

class Tensor:
    def __init__(self, data, requires_grad=False, _children=(), _op=""):
        if isinstance(data, Tensor): data = data.data
        self.data = xp.asarray(data, dtype=xp.float32)
        if not requires_grad and _children:
            requires_grad = any(c.requires_grad for c in _children if isinstance(c, Tensor))
        self.requires_grad = requires_grad
        self.grad      = xp.zeros_like(self.data) if requires_grad else None
        self._backward = lambda: None
        self._prev     = set(_children)
        self._op       = _op

    @property
    def shape(self): return self.data.shape
    @property
    def ndim(self):  return self.data.ndim
    def __repr__(self): return f"Tensor(shape={self.shape})"

    def __add__(self, other):
        if not isinstance(other, Tensor): other = Tensor(xp.asarray(other, dtype=xp.float32))
        out = Tensor(self.data + other.data, _children=(self, other), _op="+")
        def _bwd():
            if self.grad  is not None: self.grad  += _sum_to_shape(out.grad, self.shape)
            if other.grad is not None: other.grad += _sum_to_shape(out.grad, other.shape)
        out._backward = _bwd; return out
    __radd__ = __add__

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            out = Tensor(self.data * other, _children=(self,), _op="*s")
            def _bwd():
                if self.grad is not None: self.grad += out.grad * other
            out._backward = _bwd; return out
        if not isinstance(other, Tensor): other = Tensor(xp.asarray(other, dtype=xp.float32))
        out = Tensor(self.data * other.data, _children=(self, other), _op="*")
        def _bwd():
            if self.grad  is not None: self.grad  += _sum_to_shape(out.grad * other.data, self.shape)
            if other.grad is not None: other.grad += _sum_to_shape(out.grad * self.data,  other.shape)
        out._backward = _bwd; return out
    __rmul__ = __mul__

    def __neg__(self): return self * (-1.0)
    def __sub__(self, other): return self + (-other)

    def __matmul__(self, other):
        out = Tensor(self.data @ other.data, _children=(self, other), _op="@")
        def _bwd():
            if self.grad  is not None: self.grad  += _sum_to_shape(out.grad @ xp.swapaxes(other.data,-1,-2), self.shape)
            if other.grad is not None: other.grad += _sum_to_shape(xp.swapaxes(self.data,-1,-2) @ out.grad,  other.shape)
        out._backward = _bwd; return out

    def reshape(self, *shape):
        out = Tensor(self.data.reshape(*shape), _children=(self,), _op="reshape")
        def _bwd():
            if self.grad is not None: self.grad += out.grad.reshape(self.shape)
        out._backward = _bwd; return out

    def transpose(self, *axes):
        out = Tensor(xp.transpose(self.data, axes), _children=(self,), _op="T")
        def _bwd():
            if self.grad is not None:
                inv = [0]*len(axes)
                for i, a in enumerate(axes): inv[a] = i
                self.grad += xp.transpose(out.grad, inv)
        out._backward = _bwd; return out

    def softmax(self, axis=-1):
        shifted = self.data - xp.max(self.data, axis=axis, keepdims=True)
        e = xp.exp(shifted); s = e / e.sum(axis=axis, keepdims=True)
        out = Tensor(s, _children=(self,), _op="softmax")
        def _bwd():
            if self.grad is not None:
                self.grad += s * (out.grad - (s * out.grad).sum(axis=axis, keepdims=True))
        out._backward = _bwd; return out

    def gelu(self):
        x = self.data; c = xp.sqrt(2.0 / xp.pi)
        inner = c * (x + 0.044715 * x**3); t = xp.tanh(inner)
        out = Tensor(0.5 * x * (1.0 + t), _children=(self,), _op="gelu")
        def _bwd():
            if self.grad is not None:
                self.grad += out.grad * (0.5*(1+t) + 0.5*x*(1-t*t)*c*(1+3*0.044715*x*x))
        out._backward = _bwd; return out

    def sum(self, axis=None, keepdims=False):
        out = Tensor(self.data.sum(axis=axis, keepdims=keepdims), _children=(self,), _op="sum")
        def _bwd():
            if self.grad is not None:
                g = out.grad
                if axis is not None and not keepdims: g = xp.expand_dims(g, axis)
                b = xp.broadcast_to(g, self.shape)
                self.grad += b.copy() if hasattr(b, "copy") else b
        out._backward = _bwd; return out

    def mean(self, axis=None, keepdims=False):
        n = math.prod(self.data.shape) if axis is None else self.data.shape[axis]
        return self.sum(axis=axis, keepdims=keepdims) * (1.0 / n)

    def backward(self):
        order, seen = [], set()
        def topo(t):
            if id(t) not in seen:
                seen.add(id(t))
                for c in t._prev: topo(c)
                order.append(t)
        topo(self)
        self.grad = xp.ones_like(self.data)
        for t in reversed(order): t._backward()

    def zero_grad(self):
        if self.grad is not None: self.grad = xp.zeros_like(self.data)


## 4 · Layers

In [ ]:
class Module:
    def parameters(self):
        ps = []
        for v in vars(self).values():
            if isinstance(v, Tensor) and v.requires_grad: ps.append(v)
            elif isinstance(v, Module): ps.extend(v.parameters())
            elif isinstance(v, list):
                for it in v:
                    if isinstance(it, Module): ps.extend(it.parameters())
        return ps
    def zero_grad(self):
        for p in self.parameters(): p.zero_grad()

class Embedding(Module):
    def __init__(self, n, d):
        self.weight = Tensor(xp.asarray(xp.random.randn(n, d), dtype=xp.float32)*0.02, requires_grad=True)
    def __call__(self, idx):
        out = Tensor(self.weight.data[idx], _children=(self.weight,), _op="emb")
        w = self.weight
        def _bwd():
            if w.grad is not None: xp.add.at(w.grad, idx, out.grad)
        out._backward = _bwd; return out

class Linear(Module):
    def __init__(self, i, o, bias=True):
        lim = xp.sqrt(6.0 / (i + o))
        self.weight = Tensor(xp.asarray(xp.random.uniform(-float(lim), float(lim), (i, o)), dtype=xp.float32), requires_grad=True)
        self.bias   = Tensor(xp.zeros(o, dtype=xp.float32), requires_grad=True) if bias else None
    def __call__(self, x):
        out = x @ self.weight
        return out + self.bias if self.bias is not None else out

class LayerNorm(Module):
    def __init__(self, d, eps=1e-5):
        self.gain = Tensor(xp.ones(d,  dtype=xp.float32), requires_grad=True)
        self.bias = Tensor(xp.zeros(d, dtype=xp.float32), requires_grad=True)
        self.eps  = eps
    def __call__(self, x):
        mu = x.data.mean(axis=-1, keepdims=True); var = x.data.var(axis=-1, keepdims=True)
        si = 1.0 / xp.sqrt(var + self.eps); xh = (x.data - mu) * si
        out = Tensor(self.gain.data * xh + self.bias.data, _children=(x, self.gain, self.bias), _op="ln")
        g, b, D = self.gain, self.bias, x.data.shape[-1]
        def _bwd():
            if g.grad is not None: g.grad += (out.grad * xh).reshape(-1,D).sum(axis=0)
            if b.grad is not None: b.grad +=  out.grad      .reshape(-1,D).sum(axis=0)
            if x.grad is not None:
                dxh = out.grad * g.data
                x.grad += si/D*(D*dxh - dxh.sum(axis=-1,keepdims=True) - xh*(dxh*xh).sum(axis=-1,keepdims=True))
        out._backward = _bwd; return out

class MultiHeadAttention(Module):
    def __init__(self, d, h):
        assert d % h == 0; self.h = h; self.dk = d // h
        self.Wq = Linear(d,d,bias=False); self.Wk = Linear(d,d,bias=False)
        self.Wv = Linear(d,d,bias=False); self.Wo = Linear(d,d,bias=False)
    def __call__(self, x):
        B,T,D = x.shape; H,dk = self.h, self.dk
        Q = self.Wq(x).reshape(B,T,H,dk).transpose(0,2,1,3)
        K = self.Wk(x).reshape(B,T,H,dk).transpose(0,2,1,3)
        V = self.Wv(x).reshape(B,T,H,dk).transpose(0,2,1,3)
        sc = (Q @ K.transpose(0,1,3,2)) * (1.0/xp.sqrt(dk))
        sc = sc + Tensor(xp.triu(xp.ones((T,T),dtype=xp.float32),k=1)*(-1e9))
        attn = (sc.softmax(axis=-1) @ V).transpose(0,2,1,3).reshape(B,T,D)
        return self.Wo(attn)

class FeedForward(Module):
    def __init__(self, d, ff): self.fc1=Linear(d,ff); self.fc2=Linear(ff,d)
    def __call__(self, x): return self.fc2(self.fc1(x).gelu())

class TransformerBlock(Module):
    def __init__(self, d, h, ff):
        self.ln1=LayerNorm(d); self.attn=MultiHeadAttention(d,h)
        self.ln2=LayerNorm(d); self.ffn=FeedForward(d,ff)
    def __call__(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


## 5 · Transformer

In [ ]:
def sinusoidal_encoding(max_len, d_model):
    pe = _np.zeros((max_len, d_model), dtype=_np.float32)
    pos = _np.arange(max_len)[:,None]
    div = _np.exp(_np.arange(0, d_model, 2)*(-_np.log(10000.0)/d_model))
    pe[:,0::2] = _np.sin(pos*div); pe[:,1::2] = _np.cos(pos*div)
    return from_numpy(pe)

def cross_entropy_loss(logits, targets, mask=None):
    B,T,V = logits.shape
    ln = to_numpy(logits.data); tn = _np.asarray(to_numpy(targets), dtype=_np.int64)
    mn = _np.ones((B,T),dtype=_np.float32) if mask is None else _np.asarray(to_numpy(mask),dtype=_np.float32)
    sh = ln - ln.max(axis=-1,keepdims=True); pr = _np.exp(sh); pr /= pr.sum(axis=-1,keepdims=True)
    fp = pr.reshape(-1,V); ft = tn.reshape(-1); fm = mn.reshape(-1); N = max(float(fm.sum()),1.0)
    nll = -_np.log(fp[_np.arange(ft.shape[0]),ft]+1e-9)
    loss = Tensor(_np.array((nll*fm).sum()/N,dtype=_np.float32), _children=(logits,), _op="ce")
    loss.requires_grad = True; loss.grad = xp.zeros_like(loss.data)
    def _bwd():
        if logits.grad is not None:
            g=pr.copy(); gf=g.reshape(-1,V); gf[_np.arange(ft.shape[0]),ft]-=1.0
            gf*=fm[:,None]; gf/=N; logits.grad+=from_numpy(g)
    loss._backward = _bwd; return loss

class ToolCallingLM(Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=2, d_ff=512, max_seq_len=256):
        self.tok_emb  = Embedding(vocab_size, d_model)
        self.pos_enc  = sinusoidal_encoding(max_seq_len, d_model)
        self.blocks   = [TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        self.ln_final = LayerNorm(d_model)
        self.head     = Linear(d_model, vocab_size, bias=False)
    def __call__(self, ids):
        ids = from_numpy(_np.asarray(ids, dtype=_np.int64)); B,T = ids.shape
        x = self.tok_emb(ids) + Tensor(self.pos_enc[:T])
        for blk in self.blocks: x = blk(x)
        return self.head(self.ln_final(x))
    def generate(self, prompt_ids, max_new_tokens=64, eos_id=None):
        ids = list(_np.asarray(to_numpy(prompt_ids), dtype=_np.int64))
        for _ in range(max_new_tokens):
            nxt = int(to_numpy(self(_np.array([ids],dtype=_np.int64)).data)[0,-1].argmax())
            ids.append(nxt)
            if eos_id is not None and nxt == eos_id: break
        return _np.array(ids, dtype=_np.int64)


## 6 · Tokenizer

In [ ]:
SPECIAL_TOKENS = ["<PAD>","<BOS>","<EOS>","<UNK>","<CALL>","NO_CALL"]
PUNCT = set('()=,"')

def _tokenize_text(text):
    toks = []
    for w in text.split():
        buf = ""
        for ch in w:
            if ch in PUNCT:
                if buf: toks.append(buf); buf=""
                toks.append(ch)
            else: buf += ch
        if buf: toks.append(buf)
    return toks

class Tokenizer:
    def __init__(self): self.token2id={}; self.id2token={}
    @property
    def vocab_size(self): return len(self.token2id)
    @property
    def pad_id(self): return self.token2id["<PAD>"]
    @property
    def eos_id(self): return self.token2id["<EOS>"]
    @property
    def unk_id(self): return self.token2id["<UNK>"]

    def build_vocab(self, texts):
        freq={}
        for t in texts:
            for tok in _tokenize_text(t): freq[tok]=freq.get(tok,0)+1
        self.token2id={}
        for tok in SPECIAL_TOKENS: self.token2id[tok]=len(self.token2id)
        for tok,cnt in sorted(freq.items(),key=lambda x:-x[1]):
            if tok not in self.token2id: self.token2id[tok]=len(self.token2id)
        self.id2token={i:t for t,i in self.token2id.items()}; return self

    def encode(self, text, add_eos=False):
        ids=[self.token2id.get(t,self.unk_id) for t in _tokenize_text(text)]
        if add_eos: ids.append(self.eos_id); return ids

    def decode(self, ids):
        toks=[self.id2token.get(int(i),"<UNK>") for i in ids
              if self.id2token.get(int(i)) not in ("<PAD>","<BOS>","<EOS>")]
        text=" ".join(toks)
        for p in PUNCT: text=text.replace(" "+p+" ",p).replace(" "+p,p).replace(p+" ",p)
        return text

    def save(self, path):
        with open(path,"w") as f: json.dump(self.token2id,f,indent=2)

    @classmethod
    def load(cls, path):
        tok=cls()
        with open(path) as f: tok.token2id=json.load(f)
        tok.id2token={int(i):t for t,i in tok.token2id.items()}; return tok

    @classmethod
    def from_data_files(cls, *paths):
        texts=[]
        for p in paths:
            with open(p) as f: d=json.load(f)
            for ex in d: texts+=[ex["input_text"],ex["target_text"]]
        return cls().build_vocab(texts)


## 7 · Optimizer

In [ ]:
class Adam:
    def __init__(self, params, lr=1e-3, betas=(0.9,0.999), eps=1e-8, grad_clip=1.0):
        self.params=params; self.lr=lr; self.b1,self.b2=betas; self.eps=eps; self.clip=grad_clip; self.t=0
        self.m=[xp.zeros_like(p.data) for p in params]
        self.v=[xp.zeros_like(p.data) for p in params]
    def step(self):
        self.t+=1
        if self.clip>0:
            norm=xp.sqrt(sum(xp.sum(p.grad**2) for p in self.params if p.grad is not None))
            if float(norm)>self.clip:
                sc=self.clip/(float(norm)+1e-9)
                for p in self.params:
                    if p.grad is not None: p.grad*=sc
        for i,p in enumerate(self.params):
            if p.grad is None: continue
            self.m[i]=self.b1*self.m[i]+(1-self.b1)*p.grad
            self.v[i]=self.b2*self.v[i]+(1-self.b2)*p.grad**2
            mh=self.m[i]/(1-self.b1**self.t); vh=self.v[i]/(1-self.b2**self.t)
            p.data-=self.lr*mh/(xp.sqrt(vh)+self.eps)

class CosineScheduler:
    def __init__(self, opt, warmup, total): self.opt=opt; self.w=warmup; self.T=total; self.base=opt.lr
    def step(self, s):
        lr=self.base*(s+1)/self.w if s<self.w else self.base*0.5*(1+_np.cos(_np.pi*(s-self.w)/max(1,self.T-self.w)))
        self.opt.lr=lr; return lr


## 8 · Data Utilities

In [ ]:
def load_sequences(path):
    with open(path) as f: return json.load(f)

def prepare_batch(examples, tokenizer, max_len):
    cid=tokenizer.token2id["<CALL>"]
    XS,YS,MS=[],[],[]
    for ex in examples:
        ids=tokenizer.encode(ex["input_text"]+" "+ex["target_text"],add_eos=True)
        if len(ids)>max_len: ids=ids[:max_len]
        xi=ids[:-1]; yi=ids[1:]
        m=_np.zeros(len(yi),dtype=_np.float32)
        cs=[j for j,t in enumerate(xi) if t==cid]
        m[cs[-1]:]=1.0 if cs else 1.0
        XS.append(xi); YS.append(yi); MS.append(m)
    L=max(len(s) for s in XS); pad=tokenizer.pad_id
    inp=_np.full((len(examples),L),pad,dtype=_np.int64)
    tgt=_np.full((len(examples),L),pad,dtype=_np.int64)
    msk=_np.zeros((len(examples),L),dtype=_np.float32)
    for i,(x,y,m) in enumerate(zip(XS,YS,MS)):
        inp[i,:len(x)]=x; tgt[i,:len(y)]=y; msk[i,:len(m)]=m
    return inp,tgt,msk

def batches(data, bs, tokenizer, max_len, shuffle=True):
    idx=_np.arange(len(data))
    if shuffle: _np.random.shuffle(idx)
    for s in range(0,len(data),bs):
        yield prepare_batch([data[i] for i in idx[s:s+bs]], tokenizer, max_len)


## 9 · Evaluation Utilities

In [ ]:
def parse_tool_call(text):
    t=text.strip().lower()
    if t.startswith("no_call"): return ("no_call","")
    if "(" in t: idx=t.index("("); return (t[:idx].strip(), t[idx+1:].rstrip(")").strip())
    return (t,"")

def token_f1(pred, gold):
    pt=_tokenize_text(pred.lower()); gt=_tokenize_text(gold.lower())
    if not gt and not pt: return 1.0
    if not gt or  not pt: return 0.0
    ps={t:pt.count(t) for t in set(pt)}; gs={t:gt.count(t) for t in set(gt)}
    ov=sum(min(c,ps.get(t,0)) for t,c in gs.items())
    if ov==0: return 0.0
    return 2*ov/len(pt)*ov/len(gt) / (ov/len(pt)+ov/len(gt))

def evaluate_model(model, tokenizer, test_data, cfg):
    mcfg=cfg["model"]; tcfg=cfg["training"]
    print("  Generating predictions...")
    t0=time.time(); preds=[]
    for i,ex in enumerate(test_data):
        ids=tokenizer.encode(ex["input_text"])
        gen=model.generate(_np.array(ids,dtype=_np.int64),max_new_tokens=50,eos_id=tokenizer.eos_id)
        preds.append(tokenizer.decode(gen[len(ids):]))
        if (i+1)%200==0: print(f"    {i+1}/{len(test_data)}  ({time.time()-t0:.1f}s)")
    golds=[ex["target_text"] for ex in test_data]
    pp=[parse_tool_call(p) for p in preds]; pg=[parse_tool_call(g) for g in golds]

    em  = sum(p.strip().lower()==g.strip().lower() for p,g in zip(preds,golds))/len(golds)
    tsa = sum(pt==gt for (pt,_),(gt,_) in zip(pp,pg))/len(golds)
    ae=ao=0
    for (pt,pa),(gt,ga) in zip(pp,pg):
        if gt=="no_call": continue
        if pt==gt: ae+=1; ao+=pa.strip()==ga.strip()
    aem=ao/ae if ae else 0.0
    f1s=[token_f1(pa,ga) if pt==gt else 0.0 for (pt,pa),(gt,ga) in zip(pp,pg) if gt!="no_call"]
    f1=sum(f1s)/len(f1s) if f1s else 0.0

    print("  Computing perplexity...")
    tl=tt=0
    for inp,tgt,msk in batches(test_data,tcfg["batch_size"],tokenizer,mcfg["max_seq_len"],shuffle=False):
        ln=to_numpy(model(inp).data); sh=ln-ln.max(axis=-1,keepdims=True)
        pr=_np.exp(sh); pr/=pr.sum(axis=-1,keepdims=True)
        fp=pr.reshape(-1,ln.shape[-1]); ft=_np.asarray(tgt).reshape(-1); fm=_np.asarray(msk).reshape(-1)
        tl+=(-_np.log(fp[_np.arange(ft.shape[0]),ft]+1e-9)*fm).sum(); tt+=fm.sum()
    ppl=float(_np.exp(tl/max(tt,1)))

    nc=nok=0
    for (pt,_),(gt,_) in zip(pp,pg):
        if gt=="no_call": nc+=1; nok+=pt=="no_call"

    return {"EM":em,"TSA":tsa,"AEM":aem,"ArgF1":f1,"PPL":ppl,"NO_CALL":nok/nc if nc else 0.0}, preds


## 10 · Configurations

In [ ]:
CONFIGS = {
    "small":  {"model": {"vocab_size":3000,"d_model":128,"n_heads":4,"n_layers":2,"d_ff":512, "max_seq_len":256},
               "training": {"batch_size":16,"epochs":30,"learning_rate":1e-3,"warmup_steps":500,"grad_clip":1.0,"beta1":.9,"beta2":.999,"eps":1e-8}},
    "medium": {"model": {"vocab_size":3000,"d_model":192,"n_heads":6,"n_layers":3,"d_ff":768, "max_seq_len":256},
               "training": {"batch_size":16,"epochs":30,"learning_rate":8e-4,"warmup_steps":500,"grad_clip":1.0,"beta1":.9,"beta2":.999,"eps":1e-8}},
    "large":  {"model": {"vocab_size":3000,"d_model":256,"n_heads":8,"n_layers":4,"d_ff":1024,"max_seq_len":256},
               "training": {"batch_size":16,"epochs":30,"learning_rate":5e-4,"warmup_steps":500,"grad_clip":1.0,"beta1":.9,"beta2":.999,"eps":1e-8}},
}


## 11 · Load Data

In [ ]:
train_data = load_sequences(DATA_DIR / "train_sequences.json")
val_data   = load_sequences(DATA_DIR / "val_sequences.json")
test_data  = load_sequences(DATA_DIR / "test_sequences.json")

tokenizer = Tokenizer.from_data_files(
    DATA_DIR / "train_sequences.json",
    DATA_DIR / "val_sequences.json",
)
for cfg in CONFIGS.values():
    cfg["model"]["vocab_size"] = tokenizer.vocab_size

with open(DATA_DIR / "tools.json") as f: tools = json.load(f)
tool_string = ", ".join(t["name"]+"("+",".join(t["arguments"].keys())+")" for t in tools)

print(f"Train: {len(train_data):,}  Val: {len(val_data):,}  Test: {len(test_data):,}")
print(f"Vocab: {tokenizer.vocab_size:,}  |  {tool_string}")


## 12 · Training

In [ ]:
def train_model(name):
    cfg=CONFIGS[name]; mc=cfg["model"]; tc=cfg["training"]
    model=ToolCallingLM(vocab_size=mc["vocab_size"],d_model=mc["d_model"],
                        n_heads=mc["n_heads"],n_layers=mc["n_layers"],
                        d_ff=mc["d_ff"],max_seq_len=mc["max_seq_len"])
    n=sum(math.prod(p.data.shape) for p in model.parameters())
    print(f"\n{'='*55}")
    print(f"  Training: {name.upper()}  ({n:,} params)")
    print(f"{'='*55}")

    opt=Adam(model.parameters(),lr=tc["learning_rate"],betas=(tc["beta1"],tc["beta2"]),
             eps=tc["eps"],grad_clip=tc["grad_clip"])
    spe=len(train_data)//tc["batch_size"]
    sched=CosineScheduler(opt,tc["warmup_steps"],spe*tc["epochs"])

    ckpt=ROOT/"checkpoints"/name; ckpt.mkdir(parents=True,exist_ok=True)
    hist={"train_loss":[],"val_loss":[],"tool_acc":[]}
    best=float("inf"); step=0

    for ep in range(1,tc["epochs"]+1):
        tl=nb=0; t0=time.time()
        for inp,tgt,msk in batches(train_data,tc["batch_size"],tokenizer,mc["max_seq_len"]):
            loss=cross_entropy_loss(model(inp),tgt,mask=msk)
            model.zero_grad(); loss.backward(); sched.step(step); opt.step()
            tl+=float(loss.data); nb+=1; step+=1
        vl=vb=0
        for inp,tgt,msk in batches(val_data,tc["batch_size"],tokenizer,mc["max_seq_len"],shuffle=False):
            vl+=float(cross_entropy_loss(model(inp),tgt,mask=msk).data); vb+=1
        correct=sum(
            tokenizer.decode(model.generate(_np.array(tokenizer.encode(ex["input_text"]),dtype=_np.int64),
                             max_new_tokens=30,eos_id=tokenizer.eos_id)[len(tokenizer.encode(ex["input_text"])):]).split("(")[0].strip()
            == ex["target_text"].split("(")[0].strip()
            for ex in val_data[:50]
        )
        at=tl/max(nb,1); av=vl/max(vb,1); tsa=correct/50
        hist["train_loss"].append(at); hist["val_loss"].append(av); hist["tool_acc"].append(tsa)
        print(f"  Ep {ep:2d}/{tc['epochs']}  train={at:.4f}  val={av:.4f}  tool_acc={tsa:.1%}  ({time.time()-t0:.1f}s)")
        if av<best:
            best=av
            _np.savez(ckpt/"best_model.npz",**{f"p{i}":to_numpy(p.data) for i,p in enumerate(model.parameters())})
            tokenizer.save(ckpt/"tokenizer.json")
            print(f"    ✓ checkpoint saved (val={av:.4f})")
    return model, hist


In [ ]:
trained = {}   # name -> (model, history)

for _name in MODELS_TO_RUN:
    _model, _hist = train_model(_name)
    trained[_name] = (_model, _hist)

print("\nDone training:", list(trained.keys()))


## 13 · Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name, (_, hist) in trained.items():
    ep = range(1, len(hist["train_loss"])+1); c = PALETTE.get(name, "gray")
    axes[0].plot(ep, hist["train_loss"],                label=name, color=c)
    axes[1].plot(ep, hist["val_loss"],                  label=name, color=c)
    axes[2].plot(ep, [v*100 for v in hist["tool_acc"]], label=name, color=c)

axes[0].set_title("Train Loss");         axes[0].set_ylabel("Loss")
axes[1].set_title("Validation Loss")
axes[2].set_title("Tool Accuracy (%)");  axes[2].set_ylabel("%")
axes[2].yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
for ax in axes: ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.3)
plt.suptitle("Tool-Calling LM — Training Curves", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()


## 14 · Evaluate on Test Set

In [ ]:
results = {}   # name -> (metrics_dict, predictions_list)

for name, (model, _) in trained.items():
    print(f"\n=== {name.upper()} ===")
    res, preds = evaluate_model(model, tokenizer, test_data, CONFIGS[name])
    results[name] = (res, preds)


## 15 · Results Comparison

In [ ]:
METRIC_LABELS = {"EM":"Exact Match","TSA":"Tool Selection Acc",
                 "AEM":"Arg Exact Match","ArgF1":"Argument F1",
                 "PPL":"Perplexity","NO_CALL":"NO_CALL Accuracy"}

names = list(results.keys())
col_w = 11
hdr = f"  {'Metric':<23}" + "".join(f"  {n:>{col_w}}" for n in names)
sep = "=" * len(hdr)
print("\n" + sep); print(hdr); print(sep)
for key, label in METRIC_LABELS.items():
    row = f"  {label:<23}"
    for n in names:
        v = results[n][0][key]
        row += f"  {v:>{col_w}.2f}" if key=="PPL" else f"  {v:>{col_w-1}.1%} "
    print(row)
print(sep)


## 16 · Metrics Bar Chart

In [ ]:
pct_keys = ["EM","TSA","AEM","ArgF1","NO_CALL"]
names = list(results.keys())
x = _np.arange(len(pct_keys)); w = 0.8 / max(len(names), 1)

fig, ax = plt.subplots(figsize=(11, 5))
for i, name in enumerate(names):
    vals = [results[name][0][k]*100 for k in pct_keys]
    bars = ax.bar(x + i*w - (len(names)-1)*w/2, vals, w,
                  label=name, color=PALETTE.get(name,"gray"))
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                f"{v:.0f}%", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([METRIC_LABELS[k] for k in pct_keys], rotation=15, ha="right")
ax.set_ylabel("Score (%)"); ax.set_ylim(0, 115)
ax.set_title("Model Comparison — Test Set Metrics")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("metrics_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## 17 · Sample Predictions

In [ ]:
N = 10
names = list(results.keys())
print("-"*70)
print(f"  SAMPLE PREDICTIONS  (first {N} test examples)")
print("-"*70)
for i in range(min(N, len(test_data))):
    inp   = test_data[i]["input_text"]
    query = inp.split("\n")[1] if "\n" in inp else inp
    gold  = test_data[i]["target_text"]
    print(f"\n  {query}")
    print(f"    Gold: {gold}")
    for name in names:
        pred = results[name][1][i].strip()
        mark = "V" if pred.lower()==gold.lower() else "X"
        print(f"    {name:6s} [{mark}]: {pred}")
print("\n" + "-"*70)


## 18 · Live Inference Demo

In [ ]:
def predict_all(query, max_new_tokens=50):
    prompt_ids = tokenizer.encode(
        "<BOS>" + "\n" + "User: " + query + "\n" +
        "Available tools: " + tool_string + "\n" + "<CALL>"
    )
    out = {}
    for name, (model, _) in trained.items():
        ids = model.generate(_np.array(prompt_ids, dtype=_np.int64),
                             max_new_tokens=max_new_tokens, eos_id=tokenizer.eos_id)
        out[name] = tokenizer.decode(ids[len(prompt_ids):])
    return out

example_queries = [
    "What's the weather in Denver tomorrow?",
    "Set a timer for 10 minutes",
    "Play some jazz music",
    "What time is it in Tokyo?",
    "Tell me a joke",
]

print("-"*70)
for q in example_queries:
    preds = predict_all(q)
    print(f"Query: {q}")
    for name, pred in preds.items():
        print(f"  {name:6s}: {pred}")
    print("-"*70)
